# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['BKOZIJQDBG', 'YUQOLRRTBX'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 2, 11, 15, 26,  9, 10, 17,  4,  2,  7],
       [25, 21, 17, 15, 12, 18, 18, 20,  2, 24]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  7,  2,  4, 17, 10,  9, 26, 15, 11],
       [ 0, 24,  2, 20, 18, 18, 12, 15, 17, 21]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 7,  2,  4, 17, 10,  9, 26, 15, 11,  2],
       [24,  2, 20, 18, 18, 12, 15, 17, 21, 25]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    input_shape=(None,))#batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        #注意力相关层
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    def call(self, enc_ids, dec_ids):
        '''
        todo
        
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        #编码
        enc_out, enc_state = self.encode(enc_ids)   # enc_out: (batch, enc_len, hidden)
        #解码器初始状态为编码器最终状态（SimpleRNNCell 需要 states 列表）
        state = [enc_state]
        
        #准备存储每个时间步的 logits
        logits_list = []
        dec_len = tf.shape(dec_ids)[1]

        #循环解码
        for t in tf.range(dec_len):
            #当前输入 token: dec_ids[:, t]
            x = dec_ids[:, t]                     # (batch,)
            #嵌入
            x_emb = self.embed_layer(x)           # (batch, 64)
            #解码器一步
            h, [new_state] = self.decoder_cell(x_emb, state)   # h: (batch, hidden)
            state = [new_state]
            #注意力：基于当前隐藏状态 h 和编码器输出 enc_out 计算上下文向量
            context = self._attention(h, enc_out)        # (batch, hidden)
            #拼接隐藏状态和上下文，输出 logits
            combined = tf.concat([h, context], axis=-1)  # (batch, hidden*2)
            logits_t = self.dense(combined)              # (batch, v_sz)
            logits_list.append(logits_t)

        # 将时间步维度堆叠起来
        logits = tf.stack(logits_list, axis=1)           # (batch, dec_len, v_sz)
        return logits

    def _attention(self, query, enc_out):
        # query: (batch, hidden), enc_out: (batch, enc_len, hidden)
        keys = self.dense_attn(enc_out)                         # (batch, enc_len, hidden)
        scores = tf.reduce_sum(keys * tf.expand_dims(query, 1), axis=-1)  # (batch, enc_len)
        attn_weights = tf.nn.softmax(scores, axis=-1)          # (batch, enc_len)
        context = tf.reduce_sum(enc_out * tf.expand_dims(attn_weights, -1), axis=1)  # (batch, hidden)
        return context
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, enc_state
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        '''
    
        '''
        todo
        参考sequence_reversal-exercise, 自己构建单步解码逻辑'''
        #嵌入
        x_emb = self.embed_layer(x)                     # (batch, 64)
        #解码器一步
        h, [new_state] = self.decoder_cell(x_emb, [state])  # h: (batch, hidden)
        #注意力
        context = self._attention(h, enc_out)           # (batch, hidden)
        #输出层
        combined = tf.concat([h, context], axis=-1)
        logits = self.dense(combined)                   # (batch, v_sz)
        #取概率最大的 token
        out = tf.argmax(logits, axis=-1)                # (batch,)
        return out, new_state

# Loss函数以及训练逻辑

In [4]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    grads_and_vars = [(g, v) for g, v in zip(grads, model.trainable_variables) if g is not None]
    if not grads_and_vars:
        raise ValueError('No gradients found. Please check model forward path and loss.')
    optimizer.apply_gradients(grads_and_vars)
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
#先做一次前向，确保变量已创建，再进入训练--这套作业代码全是这里有问题！
_, warmup_enc_x, warmup_dec_x, _ = get_batch(2, 20)
_ = model(warmup_enc_x, warmup_dec_x)
train(model, optimizer, seqlen=20)

d:\anaconda3\envs\nndl\lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


step 0 : loss 3.2995002
step 500 : loss 1.4588289
step 1000 : loss 0.4552428
step 1500 : loss 0.15931559


<tf.Tensor: shape=(), dtype=float32, numpy=0.07992374897003174>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state)[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, False, False, False, True, False, True, True, True, True, True, True, True, True, False, True, True, True, True, False, True, True, True, True, False, True, True, False, False, True, True, True]
[('QXKOCIKOFLJWFCEVRHNS', 'SNHRVECFWJLFOKICOKXQ'), ('KZOZIYTAZBUHHTOLKMNQ', 'QNMKLOTHHUBZATYIZOZK'), ('BRKKFYXBYMQHRNXUEKOK', 'KOKEUXNRHQMYBXYFKVRB'), ('AIOXORAUPQEKDOIXQKBN', 'NBKQXIODKEQPUAROXOIA'), ('UUAANBIGVJMSVMYVGNHM', 'MUNGVYMVSMJVGIBNACUX'), ('LSGLVUNQIXLXGKWMRZZX', 'XZZRMWKGXLXIQNUVLGSL'), ('YCQZWYYIXQXXNSYXWURH', 'HRUWXYSNXXQXIYYWZQCY'), ('UNVMOVUCLCIVREGFLLEQ', 'QELLFGERVICLCUVOMVNU'), ('IBLSZFDIEIEWSLSNNDUV', 'VUDNNSLSWEIEIDFZSLBI'), ('EGUMTBRFXCWFTGWFIXWC', 'GSDSCWGTFWCXFRBTMUYE'), ('JYDSJRPCRBOYZBNCWGEP', 'PEGWCNBZYOBRCPRJSDYQ'), ('CHMXHZHZCNUANVVQRNPH', 'HPNRQVVNAUNCZHZHXMUC'), ('DBAZTQRGSFFPJKRIWRHN', 'NHRWIRKJPFFSGRQTZABD'), ('KUINJMULFDSRRAADTPMJ', 'JMPTDAARRBIFLUMJNIOK'), ('PQGGYEYKLKVLYNGHGTPC', 'CPTGHGNYFVKLKYEYGGQP'), ('KEJRFKWUXEBDEDJTPBNQ', 'QNBPTJDEDBEXUWKFRJEK'